In [2]:
!pip install langchain langgraph langsmith langchain-groq langchain_community

In [3]:
groq_api_key = "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"

In [4]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)


In [6]:
class State(TypedDict):
    message: Annotated[list, add_messages]
    improvements: list

In [7]:
def grammar_corrector(text: str) -> str:
    prompt = f"Correct the grammar in the following text without changing meaning:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content

In [8]:
def sentence_rewriter(text: str) -> str:
    prompt = f"Rewrite the text to make it clearer and more natural:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content


In [9]:
def tone_adjuster(text: str, tone: str = "formal") -> str:
    prompt = f"Rewrite the text in a {tone} tone:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content

In [10]:
def decide_tools(state: State):
    user_input = state["message"][-1].content

    improvements = []
    # Always do grammar correction and rewriting
    grammar_fixed = grammar_corrector(user_input)
    rewritten = sentence_rewriter(grammar_fixed)
    improvements.append(("Grammar Fix", grammar_fixed))
    improvements.append(("Rewritten", rewritten))

    # If user explicitly requests tone adjustment
    if "tone:" in user_input.lower():
        tone = user_input.lower().split("tone:")[-1].strip()
        toned = tone_adjuster(rewritten, tone)
        improvements.append((f"Tone Adjusted ({tone})", toned))

    return {"improvements": improvements}

In [11]:
def generate_response(state: State):
    response = "Here are the improvements I made:\n"
    for label, text in state["improvements"]:
        response += f"\n🔹 {label}:\n{text}\n"
    return {"message": [{"role": "assistant", "content": response}]}

In [12]:
workflow = StateGraph(State)
workflow.add_node("decide_tools", decide_tools)
workflow.add_node("generate_response", generate_response)

workflow.add_edge(START, "decide_tools")
workflow.add_edge("decide_tools", "generate_response")
workflow.add_edge("generate_response", END)

app = workflow.compile()

In [13]:
print("📝 AI Writing Assistant (type 'exit' to quit)\n")

📝 AI Writing Assistant (type 'exit' to quit)



In [ ]:
while True:
    user_text = input("You: ")
    if user_text.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break

    inputs = {"message": [{"role": "user", "content": user_text}]}

    for output in app.stream(inputs):
        for key, value in output.items():
            if key == "generate_response":
                print("\nAssistant:\n")
                print(value["message"][-1]["content"])
                print("-" * 50)

You: arrange

Assistant:

Here are the improvements I made:

🔹 Grammar Fix:
Arrange.

🔹 Rewritten:
Organize.

--------------------------------------------------
You: gramar

Assistant:

Here are the improvements I made:

🔹 Grammar Fix:
grammar

🔹 Rewritten:
The set of rules that govern how words and sentences are formed.

--------------------------------------------------
